## Weather Blanket Big Data Project

### By Noa Lahan

---

# Contents

Introduction

- Inspiration

Getting Data

- Research
- Filtering and Cleaning

Data Processing

Discarded Prototypes

Visualisation

Final Outcome

Links

- GitHub Repository
- Video Presentation

Bibliography

---


The project aims to process historical weather data trends, specifically in terms of global warming and its effects on the world at large, hopefully raising awareness of the phenomenon's impact on everyone's lives ([WWF, 2016a]).

The data visualisation will be in the form of a digital "weather blanket" which should allow for easy trend or anomaly spotting in the data.

A weather blanket is a long-term project popular in crochet and knitting communities in which the crafter adds one new row to the blanket every day over the course of a year, alternating the color of the current row based on that day's temperature. The end result produces a full length blanket with a variety of colors that creates a visual way to see the year's temperature changes ([Bollanos, 2024]). Some examples (sourced from [Pinterest]):

<img src="img/pin1.png" height="200" alt="Pinterest Fig 1">
<img src="img/pin2.png" height="200" alt="Pinterest Fig 2">
<img src="img/pin3.png" height="200" alt="Pinterest Fig 3">
<img src="img/pin4.png" height="200" alt="Pinterest Fig 4">

Instead of a row representing a day and the blanket representing a year, in this project the data - and therefore the blanket - contains 80 years worth of weather information. Each column represents a year, going from 1940 on the left-most column to 2020 on the right. Each year will be seperated into its 365 days (366 for leap years), from January 1st on top to December 31st on the bottom of the page.

---

[Bollanos, 2024]: https://www.handylittleme.com/temperature-blanket-patterns/
[Pinterest]: https://www.pinterest.co.uk/search/pins/?q=weather%20blanket&rs=typed
[WWF, 2016a]: https://www.wwf.org.uk/climate-change-and-global-warming


The following two code blocks install and import all necessary libraries:


In [ ]:
#! pip install openmeteo-requests
#! pip install requests-cache retry-requests numpy pandas
#! pip install matplotlib

In [ ]:
import openmeteo_requests
import requests
import requests_cache
import pandas as pd
from retry_requests import retry
import matplotlib.pyplot as plt
import math

---

# Getting Data


To determine which countries will be represented in the project I picked countries from three categories. I looked into articles covering the countries most vulnerable to climate change, the countries with the highest and lowest recorded temperatures, and Nasa's global temperature data for countries with heat abnormalities. The corresponding latitude and longitude were recorded.

### Maximum and Minimum
According to Saunders of BBC Science Focus, the hottest recorded (air) temperature was in the United States of America, at a confirmed 56.7°C (134°F) recorded in July of 1913. Specifically, this took place in California's Death Valley, at the aptly named Furnace Creek ([Saunders, 2024]).

The lowest recorded temperature took place in Antarctica. According the World Meteorological Organization, Vostok Station recorded a global lowest -89.2°C (-129°F) in July of 1983 ([World Meteorological Organization, 2025]).

### Vulnerable to Climate Change
I started by cross referencing a few articles covering the countries that are most vulnerable to climate change ([Concern Worldwide, 2024]; [Germanwatch, 2026]; [Iberdrola, 2021]; [The IRC, 2023]). Some countries that appear in all three articles are: Afghanistan, Chad, Democratic Republic of Congo (DRC in this project for short), and Somalia.

Nasa's global temperature data page has a map that visualises temperature differences from 1884 ([NASA, 2025]). I scanned through the time frame that this project covers (1940-2020) and noticed a few countries that have seen a dramatic temperature difference, either positive or negative.    
Canada's temperature had a couple of spikes, with the figure taken from the late 1990s.     
Poland similarly was added to the list for its noticebly warm temperature difference as seen in figure 2. These temperature spikes across central europe occured in the early 2000s and again in the 2020s, with Poland being selected over other countries around it for being the center of the temperature increase.     
On the other hand, Egypt and Japan were selected for their notably cold temperatures in the early to mid 1980s.     
Notice that Iceland was ignored although facing similarly cold temperatures in this time frame due to being explored in a previous iteration of this project - see "Discarded Prototypes".

<img src="img/fig1.png" height="200" alt="Nasa Canada Fig">
<img src="img/fig2.png" height="200" alt="Nasa Poland Fig">
<img src="img/fig3.png" height="200" alt="Nasa Egypt Fig">

Bringing the final country list to:

1. Afghanistan
2. Antarctica
3. Canada
4. Chad
5. DRC (Democratic Republic of Congo)
6. Egypt
7. Japan
8. Poland
9. Somalia
10. USA (California)

These were then googled for their individual coordinates which are required by the API used (Open Mateo) and saved into a dictionary as seen in the next code block. The getData() function in the code block that follows takes the response information from Open Mateos weather API (from the following code block, as labled), sorts it into Pandas data frames, and saves those into the countries dictionary that was previously mentioned.

[Concern Worldwide, 2024]: https://www.concern.net/news/countries-most-affected-by-climate-change
[Germanwatch, 2026]: https://www.germanwatch.org/en/cri
[Iberdrola, 2021]: https://www.iberdrola.com/sustainability/top-countries-most-affected-by-climate-change
[NASA, 2024]: https://climate.nasa.gov/vital-signs/global-temperature/?intent=121
[Saunders, 2024]: https://www.sciencefocus.com/planet-earth/hottest-place-on-earth
[The IRC, 2023]: https://www.rescue.org/uk/article/10-countries-risk-climate-disaster
[World Meteorological Organization, 2025]: https://wmo.int/asu-map?map=Temp_020

In [ ]:
countries = {
    "lat" : [33.94, -82.86, 63.59, 15.45, -4.04, 26.82, 36.20, 51.92, 5.15, 36.53],
    "long" : [67.71, 135.0, -115.50, 18.73, 21.76, 30.80, 138.25, 19.15, 46.19, -116.93],
    "name" : ["Afghanistan", "Antarctica", "Canada", "Chad", "DRC", "Egypt", "Japan", "Poland", "Somalia", "USA"],
    "min" : [],
    "max" : [],
    "dif" : []
}

In [ ]:
def getData(responses):
	''' 
    Parameters:
	openmateo response
	 
    Returns:
	array of daily data
    '''
	arr = []
	for i in range(len(countries["name"])):
		response = responses[i]
		print(f"Read {countries['name'][i]}, Coordinates {response.Latitude()}°N {response.Longitude()}°E")

		# Process daily data. The order of variables needs to be the same as requested.
		daily = response.Daily()
		daily_temp = daily.Variables(0).ValuesAsNumpy()

		data = {"date": pd.date_range(
			start = pd.to_datetime(daily.Time(), unit = "s", utc = False),
			end = pd.to_datetime(daily.TimeEnd(), unit = "s", utc = False),
			inclusive = "left"
		), "temp": daily_temp}

		df = pd.DataFrame(data = data)
		df.insert(1, "year", df['date'].dt.year)
		arr += [df]
	return arr

Get data from weather API open-mateo (note: initial code taken from open-mateo's documentation and changed as needed)


In [ ]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Get data for 80 years (1940-2020)
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": countries.get("lat"),
	"longitude": countries.get("long"),
	"start_date": "1940-01-01",
	"end_date": "2020-12-31",
	"daily": "temperature_2m_min" 
}
print("--- Daily Minimums ---")
responses = openmeteo.weather_api(url, params=params)
countries["min"] = getData(responses)

# repeat for max daily temperature
print("--- Daily Maximums ---")
params["daily"] = "temperature_2m_max"
responses = openmeteo.weather_api(url, params=params)
countries["max"] = getData(responses)

---

# Data Processing


Global warming refers to the global average temperatrue's increase which causes a lot of changes to the climate. These are not only felt in terms of warmth, the World Wide Fund for Nature says this can also manifest in "more severe cold weather events" ([WWF, 2016b]).  
As such, I wanted to not only see both the maximum and minimum temperatures, but also the differences between them, to hopefully spot a pattern in increasingly polarized weather over the decades.

The next few blocks of code take the data collected in the previous section, find the differences between the daily maximums and minimums for all countries, and save all of the information into CSV files for each country. The CSV files will have the year, day, daily minimum, daily maximum, and daily difference for each day; the data is saved in this particular way with the rest of the data cleaning and visualisation process in mind.

[WWF, 2016b]: https://www.wwf.org.uk/updates/here-are-10-myths-about-climate-change


In [ ]:
# find difference between max and min
for i in range(len(countries["name"])):
    min = countries["min"][i]
    max = countries["max"][i]
    diff = max.copy()
    dtemp = max["temp"] - min["temp"]
    diff["temp"] = dtemp
    countries["dif"] += [diff]

In [ ]:
# create a days column
years = countries["min"][0]["year"]
days = []
index = 0
current = ""
for i in range(len(years)):
    if years[i] == current :
        index += 1
    else :
        current = years[i]
        index = 0
    days += [index]

In [ ]:
# save each country's data into a csv file for visualisation
for i in range(len(countries["name"])):

   min = countries["min"][i]
   max = countries["max"][i]
   diff = countries["dif"][i]

   df = min.copy()
   df = df.rename(columns={'temp': 'min'})
   df["max"] = max["temp"]
   df["diff"] = diff["temp"]
   df["days"] = days

   # save to csv
   country = countries["name"][i]
   df.to_csv('lib/' + country + '.csv', index=False)

   print(country)
   graph = df.copy()
   graph = graph.drop("date", axis='columns')
   graph = graph.drop("year", axis='columns')
   graph = graph.drop("days", axis='columns')
   # print(graph)
   graph.plot(figsize=(20, 2))
   plt.show()

Visualisation needs the max and min temperatures to know what the range is. Since we already know the max will be in the USA and the min will be in Antarctica (as those were selected for being the hottest and coldest points, respectively - see "Getting Data" section) I will get the coldest of Antarctica's daily minimums and the hottest of USA's daily maximums

Instead of using code to find the same range for the temperature differences, a quick look at the graph plots shows all differences graphs lay between 0 and 20 degree difference, so I will use that


In [ ]:
print(countries["name"][1])
print(countries["min"][1].min())
print()
print(countries["name"][9])
print(countries["max"][9].max())

Also wanted to see how the yearly averages compare to all other countries. In the following block of code I take in turn each country, find the daily average, and then the yearly average. At the end I save all of the yearly averages into a pandas dataframe.


In [ ]:
# all yearly averages
averages = {}
for i in range(len(countries["name"])):
    avg = []
    min = countries["min"][i]
    max = countries["max"][i]

    sum = 0
    count = 0
    current = 1940
    for j in range(len(min["temp"])):
          if(current != min["year"][j]):
               avg += [sum/count]
               sum = 0
               count = 0
               current = min["year"][j]
          else:
               sum += ((min["temp"][j] + max["temp"][j])/2)
               count += 1
               
    averages[countries["name"][i]] = avg

avg_df = pd.DataFrame(data = averages)

In [ ]:
# min and max by country
print("Afghanistan")
print(avg_df["Afghanistan"].min())
print(avg_df["Afghanistan"].max())

print("Antarctica")
print(avg_df["Antarctica"].min())
print(avg_df["Antarctica"].max())

print("Canada")
print(avg_df["Canada"].min())
print(avg_df["Canada"].max())

print("Chad")
print(avg_df["Chad"].min())
print(avg_df["Chad"].max())

print("DRC")
print(avg_df["DRC"].min())
print(avg_df["DRC"].max())

print("Egypt")
print(avg_df["Egypt"].min())
print(avg_df["Egypt"].max())

print("Japan")
print(avg_df["Japan"].min())
print(avg_df["Japan"].max())

print("Poland")
print(avg_df["Poland"].min())
print(avg_df["Poland"].max())

print("Somalia")
print(avg_df["Somalia"].min())
print(avg_df["Somalia"].max())

print("USA")
print(avg_df["USA"].min())
print(avg_df["USA"].max())
avg_df

In [ ]:
# plot and save information
avg_df.plot(figsize=(20, 200))
plt.show()

name = []
year = []
temp = []
for country in averages:
    value = averages[country]
    for i in range(len(value)):
        name += [country]
        year += [i]
        temp += [value[i]]

data = {
    "name" : name,
    "year" : year,
    "temp" : temp,
}
df = pd.DataFrame(data = data)
df.to_csv('lib/All.csv', index=False)


In [ ]:
# Global min and max
print(df["temp"].min())
print(df["temp"].max())
df


---

# Discarded Prototypes


## Original Concept

When first exploring the concept of this project, the countries were chosen mostly at random. By using a world map, 10 countries of various climates were selected to span the globe, with some personal bias towards familiar locations.   
Rather than displaying every day of data across the years, which provides more precise information and better demonstrates daily fluctuation, the daily temperatures were processed into monthly averages and displayed instead. The resulting visualisation was more minimalistic, showing a 12 by 80 table rather than the 365 by 80 that the following iteration had.

<img src="img/oldcode.png" height=200px/>
<img src="img/old1.png" height=200px/>
<img src="img/old2.png" height=200px/></br>
(Discarded code and visualisation)

This concept had not given the option of a "mode" which the next iteration had. As such, there was no display of the daily highs, lows, or the differences between them. There was also no option to see all of the countries displayed alongside one another, like the next iteration had. This version allowed a selection of country and a selection of range, choosing whether to display monthly averages in terms of global range (lowest temperature recorded to highest temperature recorded), or a country specific range.

For example, Argentina and Iceland respectively:

Global range for both   
<img src="img/old3.png" height=200px/>
<img src="img/old4.png" height=200px/>  

Argentina in Argentina's range and Iceland in Iceland's range   
<img src="img/old5.png" height=200px/>
<img src="img/old6.png" height=200px/>

Iceland in Iceland's range and Argentina in Argentina's range   
<img src="img/old8.png" height=200px/>
<img src="img/old7.png" height=200px/>

The decision to use monthly averages glosses over a lot of anomalies that will only show up on a daily display, providing the user with less information about the data. The random selection of countries similarly is a mistake because it glosses over countries more closely affected by the global warming pattern this project aimed to show, as well as introducing personal bias by not having an objective frame for which countries should or should not be included in the final project. Overall the result gave little information and make it difficult to spot patterns in the data.

## Creative Computing Concept

Originally this project was designed to fit the standards of the final project requirements for the big data course of UAL's Creative Computing Institute. The following prototype is the final project that was submitted for review and received an A+. Having said that, due to it being limited in time, knowledge, and in the contraints imposed by the project critirea, the concept was later improved for the version you see today.

This version improved on the last by utilizing 3 drop down menus: country, mode, and global.

<img src="img/ddmenus.png" height=200px/>
<img src="img/ddcode.png" height=200px/>
</br>
(Webpage and corresponding code)

The first two correspond to individual countries' data. The two work together; the country menu is used to select which country to show (which csv file to access) and the mode menu is used to select a mode (what data to use from the csv). The mode menu has three mode: max, min, and difference. The max and min correspond to the daily temperature data and show this in a color range going from blue to red, blue being the colder temperatures and red being the hotter ones. The difference mode is a singular color range (I chose purple after initially trying black as it made the knit pattern clearer, more on that later), and the darker the color, the bigger the difference between that day's high and low temperature. In either case, the result is a 366 by 80 grid (366 days, non-leap years leave a blank space at the bottom of the grid; 80 years) in which each square corresponds to a single day. To make the weather blanket aspect, a knit pattern was added on top of the page. The pattern has 12 rows, each corresponding to a month with the top row being January and the bottom December. The days also go from top to bottom, the 1st on the top and the 31st on the bottom, while the years go from left to right, from 1940 as the leftmost column and 2020 the rightmost. Meaning the grid goes in order from January 1st, 1940 in the top left corner, to December 31st, 2020 in the bottom right.  
The third drop down menu corresponds to the global data. That one always accesses the same csv file ("All.csv") and uses it to show all the countries' yearly averages in a 10 by 80 grid (10 countries; 80 years). The Different options on the range menu (all 10 countries and an additional "Global" option) determine the range in which the data will be shown. These are taken from the min and max temperature exploration in "Data Processing" and saved into a dictionary in the website's JavaScript file (see below). The range menu determines which min and max temperatures will be used to display the data, which allows the user to see more nuanced details in the way the average yearly temperature has changed over those 80 years. The global range mostly just shows that some countries are colder/hotter than others, but choosing a specific country's data range makes the range much smaller which makes it much easier to see changes from year to year. Here the blanket is slightly different. Instead of 12 knit rows there are only 10, each representing a country.

<img src="img/canada1.png" width=500px/>
<img src="img/canada2.png" width=500px/>
<img src="img/canada3.png" width=500px/>
</br>
(Canada's daily minimums, maximums, and differences, respectively)

</br>
</br>
<img src="img/all3.png" width=500px/>
<img src="img/antarctica4.png" width=500px/>
<img src="img/egypt4.png" width=500px/>
</br>
(Global yearly average's in global range, Antarctica's range, and Epypt's range, respectively)

</br>
</br>
<img src="img/dictionary.png" width=400px/>
</br>
(minimums and maximums dictionary)

The data was modeled using D3.js, a JavaScript library for data visualisation ([D3js.org, 2024]). Using it, I made two graphing functions, one for the countries, and one for global. D3 helped in creating the grid, designating values to each slot in it, and attributing a color to that value based on the ranges. The first graphing function requires that both the country and mode menus have been assigned a value before it starts graphing. Based on the country input the function knows what csv file to read, and based on the mode input the function changes the data accessed, range, and color scheme as needed. The other function only takes the range value as an input. It changes the knit pattern as to the 10 row one, displays the country names, and uses the range input to access the right min and max from the aforementioned dictionary.

<img src="img/jscode.png" width=500px/>
</br>
(The JS code; The code taken the inputs from the selected drop down menus and runs the corresponding graphing function accordingly)

[D3js.org, 2024]: https://d3js.org/

---

# Data Criticism


While there is a lot of information that was processed in this project, I have to note that this is not all that global warming is. Referring back to the World Wide Fund for Nature article referenced in "Data Processing," it notes that "climate change makes extreme weather more frequent and intense, including heatwaves, wildfires and floods" ([WWF, 2016b]). While temperature is a big aspect of global warming, it is important to understand that it is not the only in which it affects our daily lives. All of these extreme weather events will only become more frequent as climate change progresses and the situation gets worse ([European Environment Agency, 2024]). All of those events were completely ignored in this project and only the temperature data was collected. This is to say, that even if the final data visualisation result seems like only a mild change in weather, it does not mean that this is a minor inconvenience. Global warming is a dire issue that affects all and as such all should do their part to reverse it. Imperial College has made a fun informative webpage which has more detail about things we each can do about climate change, read more about it here: ([Imperial College London, 2018]).

[European Environment Agency, 2024]: https://www.eea.europa.eu/en/topics/in-depth/extreme-weather-floods-droughts-and-heatwaves
[Imperial College London, 2018]: https://www.imperial.ac.uk/stories/climate-action/
[WWF, 2016b]: https://www.wwf.org.uk/updates/here-are-10-myths-about-climate-change


---

# Visualisation


---

# Final Outcome

The following is a collection of screenshots from all of the visualisations and the csv files.


### Individual countries
All in alphabetical order: Afghanistan, Antarctica, Canada, Chad, DRC, Egypt, Japan, Poland, Somalia, USA

CSV Files

<img src="img/afghanistan.png" height=100px/>
<img src="img/antarctica.png" height=100px/>
<img src="img/canada.png" height=100px/>
<img src="img/chad.png" height=100px/>
<img src="img/drc.png" height=100px/>
<img src="img/egypt.png" height=100px/>
<img src="img/japan.png" height=100px/>
<img src="img/poland.png" height=100px/>
<img src="img/somalia.png" height=100px/>
<img src="img/usa.png" height=100px/>

Daily Minimums

<img src="img/afghanistan1.png" height=100px/>
<img src="img/antarctica1.png" height=100px/>
<img src="img/canada1.png" height=100px/>
<img src="img/chad1.png" height=100px/>
<img src="img/drc1.png" height=100px/>
<img src="img/egypt1.png" height=100px/>
<img src="img/japan1.png" height=100px/>
<img src="img/poland1.png" height=100px/>
<img src="img/somalia1.png" height=100px/>
<img src="img/usa1.png" height=100px/>

Daily Maximums

<img src="img/afghanistan2.png" height=100px/>
<img src="img/antarctica2.png" height=100px/>
<img src="img/canada2.png" height=100px/>
<img src="img/chad2.png" height=100px/>
<img src="img/drc2.png" height=100px/>
<img src="img/egypt2.png" height=100px/>
<img src="img/japan2.png" height=100px/>
<img src="img/poland2.png" height=100px/>
<img src="img/somalia2.png" height=100px/>
<img src="img/usa2.png" height=100px/>

Daily Differences

<img src="img/afghanistan3.png" height=100px/>
<img src="img/antarctica3.png" height=100px/>
<img src="img/canada3.png" height=100px/>
<img src="img/chad3.png" height=100px/>
<img src="img/drc3.png" height=100px/>
<img src="img/egypt3.png" height=100px/>
<img src="img/japan3.png" height=100px/>
<img src="img/poland3.png" height=100px/>
<img src="img/somalia3.png" height=100px/>
<img src="img/usa3.png" height=100px/>

### All countries

CSV files of all and global range
</br>
<img src="img/all1.png" height=100px/>
<img src="img/all4.png" height=100px/>
<img src="img/all3.png" height=100px/>
</br>

Individual countries' range in alphabetical order: Afghanistan, Antarctica, Canada, Chad, DRC, Egypt, Japan, Poland, Somalia, USA

<img src="img/afghanistan4.png" height=100px/>
<img src="img/antarctica4.png" height=100px/>
<img src="img/canada4.png" height=100px/>
<img src="img/chad4.png" height=100px/>
<img src="img/drc4.png" height=100px/>
<img src="img/egypt4.png" height=100px/>
<img src="img/japan4.png" height=100px/>
<img src="img/poland4.png" height=100px/>
<img src="img/somalia4.png" height=100px/>
<img src="img/usa4.png" height=100px/>

---

# Links


Git hub: https://git.arts.ac.uk/23043904/bigdata/tree/main/project  
Video walkthrough: https://youtu.be/_AtRaT_nCmU


---

# Bibliography


‌Bollanos, L. (2024). 20 Temperature Blanket Patterns. [online] Handy Little Me. Available at: https://www.handylittleme.com/temperature-blanket-patterns/ [Accessed 7 Jun. 2024].

‌Concern Worldwide. (2024). The 10 countries most affected by climate change in 2024. [online] Available at: https://www.concern.net/news/countries-most-affected-by-climate-change [Accessed 7 Jun. 2024].

D3js.org. (2024). D3 by Observable | The JavaScript library for bespoke data visualization. [online] Available at: https://d3js.org/ [Accessed 7 Jun. 2024].

‌European Environment Agency (2024). Extreme weather: floods, droughts and heatwaves. [online] Europa.eu. Available at: https://www.eea.europa.eu/en/topics/in-depth/extreme-weather-floods-droughts-and-heatwaves [Accessed 7 Jun. 2024].

Germanwatch (2025). Climate Risk Index 2026. [online] Germanwatch.org. Available at: https://www.germanwatch.org/en/cri [Accessed 8 Mar. 2026].

‌Iberdrola (2021). COUNTRIES MOST AFFECTED BY CLIMATE CHANGE. [online] Iberdrola. Available at: https://www.iberdrola.com/sustainability/top-countries-most-affected-by-climate-change [Accessed 7 Jun. 2024].

‌Imperial College London (2018). 9 things you can do about climate change. [online] Imperial.ac.uk. Available at: https://www.imperial.ac.uk/stories/climate-action/ [Accessed 7 Jun. 2024].

‌NASA (2025). Global Surface Temperature | NASA Global Climate Change. [online] Climate Change: Vital Signs of the Planet. Available at: https://climate.nasa.gov/vital-signs/global-temperature/?intent=121 [Accessed 8 Mar. 2026].

Pinterest. (2024). Weather Blanket Search. [online] Available at: https://www.pinterest.co.uk/search/pins/?q=weather%20blanket&rs=typed [Accessed 7 Jun. 2024].

Saunders, T. (2024). Top 10 hottest places on Earth 2025. [online] BBC Science Focus. Available at: https://www.sciencefocus.com/planet-earth/hottest-place-on-earth [Accessed 11 Mar. 2026].

‌The IRC. (2023). 10 countries at risk of climate disaster. [online] Available at: https://www.rescue.org/uk/article/10-countries-risk-climate-disaster [Accessed 7 Jun. 2024].

World Meteorological Organization (2025). Lowest Temperature: -89.2°C (-128.6°F). [online] World Meteorological Organization. Available at: https://wmo.int/asu-map?map=Temp_020 [Accessed 11 Mar. 2026].

‌‌WWF. (2016a). What are climate change and global warming? [online] Available at: https://www.wwf.org.uk/climate-change-and-global-warming [Accessed 7 Jun. 2024].

‌WWF. (2016b). Here are 10 myths about climate change. [online] Available at: https://www.wwf.org.uk/updates/here-are-10-myths-about-climate-change [Accessed 7 Jun. 2024].

‌
